<a href="https://colab.research.google.com/github/pesco13/Python-para-Quimica/blob/main/notebooks/03_Graficas_para_quimica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 03 · Gráficas para química

**Objetivos**
* Usar `numpy` para operar con series de números sin escribir bucles.
* Construir una figura con `fig, ax = plt.subplots()` y controlar cada parte.
* Poner barras de error, anotar un punto y guardar la figura en calidad de informe.

En las prácticas anteriores los gráficos salían con `df.plot()`. Eso sirve para mirar los datos, pero una figura que va a un informe necesita ejes con unidades, barras de error y una resolución decente. Aquí se aprende a construirla pieza por pieza.

## 1. `numpy`: operar con muchos números a la vez

Una lista de Python guarda cosas; un **arreglo** de `numpy` además sabe hacer cuentas. Si multiplica una lista por 2, la lista se repite; si multiplica un arreglo por 2, se duplica cada número.

In [ ]:
import numpy as np

temperaturas_lista = [0, 10, 20, 30, 40]
temperaturas_arreglo = np.array([0, 10, 20, 30, 40])

print("lista   x 2 :", temperaturas_lista * 2)
print("arreglo x 2 :", temperaturas_arreglo * 2)

Eso se llama **vectorización**: la operación se aplica a todos los elementos a la vez. Por eso en química conviene guardar los datos en arreglos: convertir °C a K, calcular absorbancias o pasar de grados a radianes se escribe en una línea.

In [ ]:
# Solubilidad del KNO3 y del NaCl en agua (g de sal por 100 g de agua)
T_C = np.array([0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100])
KNO3 = np.array([13.3, 20.9, 31.6, 45.8, 63.9, 85.5, 110.0, 138.0, 169.0, 202.0, 246.0])
NaCl = np.array([35.7, 35.8, 36.0, 36.3, 36.6, 37.0, 37.3, 37.8, 38.4, 39.0, 39.8])

T_K = T_C + 273.15                 # una sola línea convierte las once temperaturas
print("Temperaturas en K:", T_K)
print("Solubilidad media del KNO3:", KNO3.mean().round(1), "g/100 g")
print("Máxima y mínima del NaCl :", NaCl.max(), NaCl.min())

Funciones útiles: `np.linspace(a, b, n)` genera `n` valores igualmente espaciados entre `a` y `b` (sirve para dibujar una curva suave), `np.log` y `np.exp` trabajan sobre todo el arreglo, y `.mean()`, `.std()`, `.max()` resumen.

In [ ]:
x_suave = np.linspace(0, 100, 200)   # 200 puntos entre 0 y 100
print(x_suave[:5], "...", x_suave[-1])
print("Desviación estándar del NaCl:", NaCl.std(ddof=1).round(2))

`ddof=1` pide la desviación estándar **muestral**, que es la que se usa en el laboratorio cuando se tienen unas pocas réplicas. Sin ese argumento `numpy` calcula la poblacional, que subestima la dispersión.

## 2. La anatomía de una figura

En `matplotlib` una figura (`fig`) es la hoja de papel y un eje (`ax`) es cada gráfico dibujado sobre ella. Se crean juntos:

```python
fig, ax = plt.subplots()
```

A partir de ahí todo se le pide al eje: `ax.plot(...)`, `ax.set_xlabel(...)`, `ax.legend()`. Esta forma de trabajar es la que permite después poner varios gráficos en la misma figura.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(T_C, KNO3, "o-", label="KNO$_3$")
ax.plot(T_C, NaCl, "s--", label="NaCl")

ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("Solubilidad (g por 100 g de agua)")
ax.set_title("La solubilidad del KNO$_3$ depende mucho más de la temperatura")
ax.legend()

plt.show()

Tres cosas que esa figura ya hace bien y que casi siempre faltan:

1. **Las etiquetas de los ejes dicen qué es y en qué unidad.** Un eje que dice solo `y` no sirve para nada.
2. **El título dice la conclusión**, no repite el nombre de los ejes. Compare "Solubilidad vs temperatura" con el título de arriba.
3. **Las fórmulas van en formato matemático.** Entre signos de dólar, `$KNO_3$` se escribe con el 3 como subíndice.

Los estilos de línea y marcador se combinan en una cadena corta: `"o-"` es círculo con línea continua, `"s--"` cuadrado con línea discontinua, `"^:"` triángulo con puntos, `"o"` solo marcadores.

## 3. Barras de error

Si midió por triplicado, la figura debe mostrarlo. `ax.errorbar` dibuja el promedio y una barra con la dispersión.

In [ ]:
# Absorbancia de cinco patrones, medidos por triplicado
conc = np.array([0.0, 2.0, 4.0, 6.0, 8.0])
replicas = np.array([
    [0.004, 0.007, 0.002],
    [0.168, 0.175, 0.171],
    [0.339, 0.332, 0.345],
    [0.502, 0.515, 0.498],
    [0.661, 0.670, 0.683],
])

promedio = replicas.mean(axis=1)          # axis=1 promedia cada fila
desviacion = replicas.std(axis=1, ddof=1)

print("Promedios  :", promedio.round(4))
print("Desviaciones:", desviacion.round(4))

`axis=1` recorre las columnas y deja un valor por fila (un promedio por patrón). Con `axis=0` haría lo contrario: un valor por réplica. Equivocarse de eje es uno de los errores más comunes al empezar.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.errorbar(conc, promedio, yerr=desviacion,
            fmt="o", capsize=4, color="#1b6ca8",
            label="patrones (n = 3)")

ax.set_xlabel("Concentración (mg L$^{-1}$)")
ax.set_ylabel("Absorbancia a 525 nm")
ax.set_title("La respuesta es lineal en todo el intervalo medido")
ax.legend(frameon=False)

plt.show()

`yerr` acepta la desviación estándar, el error estándar o el intervalo de confianza: elija uno y **diga cuál es** en el pie de figura. `capsize` es el ancho de las tapas de la barra; sin él las barras quedan como simples líneas.

## 4. Dos magnitudes distintas en la misma figura

Cuando las dos series tienen unidades o magnitudes muy diferentes, un segundo eje vertical permite verlas juntas. Se crea con `ax.twinx()`.

In [ ]:
presion_vapor = np.array([0.6, 1.2, 2.3, 4.2, 7.4, 12.3, 19.9, 31.2, 47.4, 70.1, 101.3])  # kPa

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(T_C, KNO3, "o-", color="#1b6ca8")
ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("Solubilidad del KNO$_3$ (g/100 g)", color="#1b6ca8")
ax.tick_params(axis="y", labelcolor="#1b6ca8")

ax2 = ax.twinx()
ax2.plot(T_C, presion_vapor, "s--", color="#d1495b")
ax2.set_ylabel("Presión de vapor del agua (kPa)", color="#d1495b")
ax2.tick_params(axis="y", labelcolor="#d1495b")

ax.set_title("Las dos magnitudes crecen con la temperatura")
plt.show()

Observe que cada eje lleva el color de su serie: sin eso el lector no sabe qué curva leer en qué escala.

Un aviso: el eje doble **sugiere** una relación entre las dos curvas que puede no existir, porque las escalas se eligen a conveniencia. Úselo cuando las dos magnitudes comparten el eje x por una razón física, y no para apilar curvas sin relación.

## 5. Señalar un punto concreto

`ax.annotate` escribe un texto y, si se le da `xytext` y `arrowprops`, lo une con una flecha al punto.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T_C, KNO3, "o-", color="#1b6ca8")

i = int(np.argmax(KNO3))          # posición del valor máximo
ax.annotate(f"{KNO3[i]:.0f} g/100 g a {T_C[i]:.0f} °C",
            xy=(T_C[i], KNO3[i]),
            xytext=(55, 215),
            arrowprops=dict(arrowstyle="->", color="0.3"))

ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("Solubilidad del KNO$_3$ (g/100 g)")
ax.set_xlim(-5, 115)
ax.set_title("A 100 °C el agua disuelve casi veinte veces más KNO$_3$ que a 0 °C")
plt.show()

`np.argmax` devuelve la **posición** del máximo, no el valor. Es la forma de encontrar el punto que se quiere anotar sin buscarlo a ojo.

Anote como mucho dos o tres puntos por figura: lo demás se lee en los ejes.

## 6. Guardar la figura

`fig.savefig` escribe el archivo. Dos detalles que marcan la diferencia:

* `dpi=300` da la resolución que piden las revistas y los informes impresos. Por defecto son 100, que se ve pixelado.
* `bbox_inches="tight"` recorta el margen blanco sobrante y evita que se corten las etiquetas.

Para una figura que se va a imprimir o escalar, guarde además un **PDF**: es vectorial y no pierde nitidez.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(conc, promedio, yerr=desviacion, fmt="o", capsize=4, color="#1b6ca8")
ax.set_xlabel("Concentración (mg L$^{-1}$)")
ax.set_ylabel("Absorbancia a 525 nm")
ax.set_title("Curva de calibración del analito")

fig.savefig("calibracion.png", dpi=300, bbox_inches="tight")
fig.savefig("calibracion.pdf", bbox_inches="tight")

import os
print("Archivos guardados:", [f for f in os.listdir() if f.startswith("calibracion")])

En Colab los archivos quedan en la máquina temporal. Para bajarlos al computador:

```python
from google.colab import files
files.download("calibracion.png")
```

Esa línea solo funciona dentro de Colab; si corre el notebook en otro sitio, dará error.

## Autocomprobación

Si las tres celdas siguientes corren sin error, lo anterior quedó bien.

In [ ]:
assert isinstance(T_K, np.ndarray) and abs(T_K[0] - 273.15) < 1e-9, "T_K debe ser el arreglo en kelvin"
assert promedio.shape == (5,), "promedio debe tener un valor por patrón: use axis=1"
assert abs(promedio[1] - 0.171333) < 1e-4, "revise el promedio del segundo patrón"
assert os.path.exists("calibracion.png"), "la figura no se guardó"
print("Todo correcto.")

## Ejercicios

**1.** Dibuje la solubilidad del NaCl frente a la temperatura en una figura propia, con ejes etiquetados y un título que diga la conclusión.

<details><summary><b>Ver solución</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T_C, NaCl, "s-", color="#6a8f3c")
ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("Solubilidad del NaCl (g por 100 g de agua)")
ax.set_title("La solubilidad del NaCl casi no cambia con la temperatura")
plt.show()
```
</details>

**2.** Calcule cuánto aumenta la solubilidad del KNO₃ por cada grado, es decir la diferencia entre valores consecutivos dividida por la diferencia de temperatura. Use `np.diff`.

<details><summary><b>Ver solución</b></summary>

```python
pendiente = np.diff(KNO3) / np.diff(T_C)
print(pendiente.round(2))
```
`np.diff` devuelve un elemento menos que el arreglo original, porque hay un intervalo menos que puntos.
</details>

**3.** Haga una figura con dos paneles lado a lado, el KNO₃ en el izquierdo y el NaCl en el derecho, con el mismo intervalo en el eje vertical.

<details><summary><b>Ver solución</b></summary>

```python
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
ax1.plot(T_C, KNO3, "o-"); ax1.set_title("KNO$_3$")
ax2.plot(T_C, NaCl, "s-"); ax2.set_title("NaCl")
for a in (ax1, ax2):
    a.set_xlabel("Temperatura (°C)")
ax1.set_ylabel("Solubilidad (g/100 g)")
fig.tight_layout()
plt.show()
```
`sharey=True` fuerza la misma escala vertical: sin eso la comparación engaña.
</details>

**4.** Tome la tabla de compuestos del curso y grafique la masa molar frente al número de átomos, anotando el compuesto más pesado.

<details><summary><b>Ver solución</b></summary>

```python
import pandas as pd
URL = "https://pesco13.github.io/Python-para-Quimica/datos/pubchem_respaldo.csv"
tabla = pd.read_csv(URL)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(tabla["TPSA"], tabla["MolecularWeight"], "o", alpha=0.6)
j = tabla["MolecularWeight"].idxmax()
ax.annotate(tabla.loc[j, "nombre"],
            xy=(tabla.loc[j, "TPSA"], tabla.loc[j, "MolecularWeight"]),
            xytext=(10, -20), textcoords="offset points",
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("TPSA (Å$^2$)")
ax.set_ylabel("Masa molar (g mol$^{-1}$)")
plt.show()
```
</details>